# Egyptian Real Estate EDA

### setup

In [1]:
import os
import sys

# 1. Set the IP to localhost to avoid the "Invalid Spark URL" error
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 2. Point to your Spark and Hadoop folders
os.environ['SPARK_HOME'] = r'C:\spark-4.1.1-bin-hadoop3\spark-4.1.1-bin-hadoop3'
os.environ['HADOOP_HOME'] = r'C:\Hadoop' 
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Add PySpark to your sys.path
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python'))
sys.path.append(os.path.join(os.environ['SPARK_HOME'], 'python', 'lib', 'py4j-0.10.9.9-src.zip'))

# 4. Initialize Spark
from pyspark import SparkConf
from pyspark.sql import SparkSession
conf = SparkConf()
conf.set("spark.python.worker.timeout", "1200") # Set to 20 minutes
conf.set("spark.driver.memory", "4g")
conf.set("spark.executor.memory", "2g")
# If using a local-mode or single-node K8s, sometimes this helps:
conf.set("spark.driver.bindAddress", "127.0.0.1")
spark = SparkSession.builder \
    .appName("JupyterSparkTest") \
    .master("local[*]") \
    .config(conf=conf) \
    .getOrCreate()

sc = spark.sparkContext

### Load And Inspect Dataset

In [2]:
import pandas as pd
import numpy as np

In [13]:
# Define the path to your Egypt Property Prices dataset
PATH = "data/raw/propertyFinder.csv"

# Load the data into a Spark DataFrame
# inferSchema=True: Automatically detects data types (integers, strings, etc.)
df_raw = spark.read.csv(PATH, header=True, inferSchema=True)

# Print basic info to verify ingestion (Project Pipeline Step II)
print(f"Total Records: {df_raw.count():,}")
print(f"Total Columns: {len(df_raw.columns)}")
df_raw.printSchema()
df_raw.show(5)

Total Records: 39,713
Total Columns: 53
root
 |-- listing_id: string (nullable = true)
 |-- internal_id: double (nullable = true)
 |-- category: string (nullable = true)
 |-- listing_type: string (nullable = true)
 |-- detail_url: string (nullable = true)
 |-- property_type: string (nullable = true)
 |-- offering_type: string (nullable = true)
 |-- completion_status: string (nullable = true)
 |-- title: string (nullable = true)
 |-- price_egp: string (nullable = true)
 |-- price_period: string (nullable = true)
 |-- price_currency: string (nullable = true)
 |-- location_full: string (nullable = true)
 |-- city: string (nullable = true)
 |-- town: string (nullable = true)
 |-- district: string (nullable = true)
 |-- subdistrict: string (nullable = true)
 |-- lat: string (nullable = true)
 |-- lon: string (nullable = true)
 |-- bedrooms: string (nullable = true)
 |-- bathrooms: string (nullable = true)
 |-- area_value: double (nullable = true)
 |-- area_unit: string (nullable = true)
 |-

### Missing counts 

In [ ]:
from pyspark.sql.functions import col, count, when
import pandas as pd
import builtins

# Calculate missing counts for every column using Spark
total_rows = df_raw.count()
missing_data = []

missing_counts = df_raw.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_raw.columns
])

counts_row = missing_counts.collect()[0]

for column in df_raw.columns:
    count_val = int(counts_row[column])
    if count_val > 0:
        percentage = builtins.round((count_val / total_rows) * 100, 1)
        missing_data.append((column, count_val, percentage))

# Create Pandas DataFrame and sort by missing count
result_df = pd.DataFrame(missing_data, columns=["Column", "Missing Count", "Percentage"])
result_df = result_df.sort_values("Missing Count", ascending=False).reset_index(drop=True)

print(result_df)

Py4JJavaError: An error occurred while calling o4579.collectToPython.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 5 in stage 71.0 failed 1 times, most recent failure: Lost task 5.0 in stage 71.0 (TID 352) (kubernetes.docker.internal executor driver): org.apache.spark.SparkException: Python worker exited unexpectedly (crashed). Consider setting 'spark.sql.execution.pyspark.udf.faulthandler.enabled' or'spark.python.worker.faulthandler.enabled' configuration to 'true' for the better Python traceback.
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:678)
	at org.apache.spark.api.python.BasePythonRunner$ReaderIterator$$anonfun$1.applyOrElse(PythonRunner.scala:663)
	at scala.runtime.AbstractPartialFunction.apply(AbstractPartialFunction.scala:35)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scala:1034)
	at org.apache.spark.api.python.PythonRunner$$anon$3.read(PythonRunner.scal